# FSA/OWI — dump metadanych v2 (Google Drive, szybki)

## Ulepszenia względem v1

- **Zapis na Google Drive** — przetrwa restart środowiska Colab
- **Wznawianie z Drive** — odczytuje postęp, kontynuuje od ostatniego miejsca
- **c=1000 zamiast 100** — 10× mniej żądań (dokumentacja LoC dopuszcza do 1000/stronę)
- **Facetowanie po dacie** — omija limit deep paging 100k (kolekcja ma ~175k)
- **Bez kolumny `reproductions`** — usuwana (za dużo zbędnych danych)
- **Zapis co 10 żądań** — minimalna utrata przy przerwaniu

## Ważne fakty o API (z dokumentacji LoC)

- **Limit: 20 żądań/minutę**, przekroczenie = blokada na 1 godzinę
- **Wielowątkowość NIE pomoże** — limit jest per-IP po stronie serwera, a zadanie jest I/O-bound (czekanie na sieć, nie liczenie). Więcej rdzeni = ten sam limit = ryzyko bana.
- **Prawdziwe przyspieszenie**: c=1000 (większe porcje) + facetowanie. Z ~1.5h robi się ~10-15 min.
- **Deep paging**: nie można stronicować poza 100k elementów. Dlatego dzielimy po latach (każdy rok <100k).

## Szacowany czas

~175k rekordów ÷ 1000 = ~175 żądań. Przy bezpiecznym tempie ~18/min ≈ **10-15 minut**.


## 0. Montowanie Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import time
import json
import csv
import http.client
from pathlib import Path

import requests
import urllib3
import pandas as pd
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# ============================================================
# KONFIGURACJA — wszystko zapisywane na Google Drive
# ============================================================
DRIVE_DIR = Path('/content/drive/MyDrive/fsa_data')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)

CSV_PATH = DRIVE_DIR / 'fsa_all_metadata.csv'
PROGRESS_PATH = DRIVE_DIR / 'scrape_progress.json'

# Rate limiting — API LoC: 20/min. Margines: ~18/min = 3.3s.
REQUEST_DELAY = 3.3
# c=500 (nie 1000): mniejsze odpowiedzi = mniej IncompleteRead przy ~9MB JSON.
RESULTS_PER_PAGE = 500

DROP_COLUMNS = {'reproductions'}

HEADERS = {
    'User-Agent': 'DecisiveMoment-Research/1.0 (academic project; contact: your-email@example.com)'
}

FSA_COLLECTION_URL = 'https://www.loc.gov/collections/fsa-owi-black-and-white-negatives/'
YEARS = list(range(1935, 1945))

# Session z connection-level retry — ponawia zerwane połączenia niżej (urllib3)
def make_session():
    s = requests.Session()
    retry = Retry(
        total=5, read=5, connect=5,
        backoff_factor=2,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=['GET'],
    )
    adapter = HTTPAdapter(max_retries=retry, pool_connections=4, pool_maxsize=4)
    s.mount('https://', adapter)
    s.mount('http://', adapter)
    return s

SESSION = make_session()

print('Konfiguracja:')
print(f'  CSV:        {CSV_PATH}')
print(f'  Per page:   {RESULTS_PER_PAGE} (zmniejszone z 1000 — niezawodność)')
print(f'  Delay:      {REQUEST_DELAY}s (~{60/REQUEST_DELAY:.0f}/min, limit 20/min)')
print(f'  Pomijane:   {DROP_COLUMNS}')
print(f'  Lata:       {YEARS[0]}-{YEARS[-1]}')


In [ ]:
def loc_api_get(url, params=None, max_retries=6):
    """Odpytuje API loc.gov z rate limiting + retry odporny na IncompleteRead.
    
    Łapie zerwany transfer dużych odpowiedzi (ChunkedEncodingError,
    ProtocolError, IncompleteRead) i ponawia, zamiast wywalać traceback.
    """
    if params is None:
        params = {}
    params['fo'] = 'json'

    for attempt in range(max_retries):
        try:
            time.sleep(REQUEST_DELAY)
            resp = SESSION.get(url, params=params, headers=HEADERS, timeout=(15, 90))
            ctype = resp.headers.get('content-type', '')

            if resp.status_code == 200:
                if 'json' in ctype:
                    return resp.json()  # IncompleteRead może paść TU — łapiemy niżej
                wait = 60 * (attempt + 1)
                print(f'    ⚠ Nie-JSON (możliwa CAPTCHA), czekam {wait}s...')
                time.sleep(wait)
            elif resp.status_code == 429:
                wait = 120 * (attempt + 1)
                print(f'    ⚠ 429 rate limit, czekam {wait}s...')
                time.sleep(wait)
            else:
                print(f'    ⚠ HTTP {resp.status_code} (próba {attempt+1})')
                time.sleep(REQUEST_DELAY * 3)

        except (requests.exceptions.ChunkedEncodingError,
                requests.exceptions.ConnectionError,
                requests.exceptions.Timeout,
                urllib3.exceptions.ProtocolError,
                urllib3.exceptions.IncompleteRead,
                http.client.IncompleteRead) as e:
            # Zerwany transfer dużej odpowiedzi — najczęstszy przypadek przy c=500/1000
            wait = REQUEST_DELAY * (attempt + 2)
            print(f'    ⚠ Zerwany transfer ({type(e).__name__}), ponawiam za {wait:.0f}s...')
            time.sleep(wait)
        except requests.exceptions.RequestException as e:
            print(f'    ⚠ Inny błąd: {e} (próba {attempt+1})')
            time.sleep(REQUEST_DELAY * 3)

    print(f'    ✗ Nieudane po {max_retries} próbach')
    return None

print('loc_api_get (odporny na IncompleteRead) gotowy')


## 1. Funkcje pomocnicze — spłaszczanie, postęp, zapis


In [ ]:
def flatten_record(rec):
    """Spłaszcza rekord do płaskiego dicta. Pomija DROP_COLUMNS."""
    flat = {}
    for k, v in rec.items():
        if k in DROP_COLUMNS:
            continue  # pomijamy np. reproductions
        if isinstance(v, (list, dict)):
            flat[k] = json.dumps(v, ensure_ascii=False)
        elif v is None:
            flat[k] = ''
        else:
            flat[k] = str(v)
    return flat

def load_progress():
    """Wczytuje postęp z Drive. Struktura: które (rok, strona) już zrobione."""
    if PROGRESS_PATH.exists():
        with open(PROGRESS_PATH) as f:
            p = json.load(f)
            # done_years_pages: lista [rok, strona] zakończonych
            p['done'] = set(tuple(x) for x in p.get('done', []))
            p['all_keys'] = set(p.get('all_keys', []))
            p['seen_ids'] = set(p.get('seen_ids', []))
            return p
    return {'done': set(), 'all_keys': set(), 'seen_ids': set(), 'total_records': 0}

def save_progress(prog):
    """Zapisuje postęp na Drive (sety → listy dla JSON)."""
    serializable = {
        'done': [list(x) for x in prog['done']],
        'all_keys': sorted(prog['all_keys']),
        'seen_ids': list(prog['seen_ids']),
        'total_records': prog['total_records'],
    }
    # Zapis atomowy: najpierw temp, potem rename (chroni przed uszkodzeniem przy przerwaniu)
    tmp = PROGRESS_PATH.with_suffix('.json.tmp')
    with open(tmp, 'w') as f:
        json.dump(serializable, f)
    tmp.replace(PROGRESS_PATH)

def append_to_csv(records, all_keys):
    """Dopisuje rekordy do CSV na Drive z pełnym zestawem kolumn."""
    if not records:
        return
    file_exists = CSV_PATH.exists()
    df = pd.DataFrame(records)
    # Uzupełnij brakujące kolumny
    for key in all_keys:
        if key not in df.columns:
            df[key] = ''
    df = df[sorted(all_keys)]
    df.to_csv(CSV_PATH, mode='a', header=not file_exists, index=False, quoting=csv.QUOTE_ALL)

print('Funkcje pomocnicze gotowe')


## 2. Sprawdź liczebność per rok (facetowanie)

Dzielimy kolekcję po latach, by ominąć limit deep paging (100k). Sprawdzamy ile rekordów ma każdy rok.


In [ ]:
print('Sprawdzam liczebność per rok...\n')
year_counts = {}

for year in YEARS:
    data = loc_api_get(FSA_COLLECTION_URL, params={
        'dates': f'{year}/{year}',
        'c': 1,
        'at': 'pagination'
    })
    if data and 'pagination' in data:
        cnt = data['pagination'].get('of', 0)
        year_counts[year] = cnt
        print(f'  {year}: {cnt:>7} rekordów')
    else:
        year_counts[year] = 0
        print(f'  {year}: ? (brak danych)')

total = sum(year_counts.values())
print(f'\n  RAZEM: {total} rekordów')
print(f'  (uwaga: sumy facetów mogą się nakładać — deduplikacja po id przy zapisie)')

# Ostrzeżenie jeśli któryś rok > 100k (deep paging limit)
for year, cnt in year_counts.items():
    if cnt > 100000:
        print(f'  ⚠ {year} ma {cnt} > 100k — trzeba dodatkowego podziału (np. po miesiącu)')


## 3. Główny scraper — facetowanie po latach, zapis na Drive

Iteruje po latach, w każdym roku stronicuje po 1000. Zapisuje co 10 żądań. Wznawialny — pomija już zrobione (rok, strona).


In [ ]:
# Wczytaj postęp z Drive
progress = load_progress()
print(f'Wznawiam. Już pobrane: {progress["total_records"]} rekordów, '
      f'{len(progress["done"])} (rok,strona) zakończonych\n')

buffer = []
requests_since_save = 0
SAVE_EVERY = 10   # zapis co 10 żądań

def flush(progress, buffer):
    """Zapisuje buffer do CSV i postęp na Drive."""
    if buffer:
        append_to_csv(buffer, progress['all_keys'])
    save_progress(progress)

try:
    for year in YEARS:
        if year_counts.get(year, 0) == 0:
            continue
        
        page = 1
        while True:
            # Pomiń jeśli ta (rok, strona) już zrobiona
            if (year, page) in progress['done']:
                page += 1
                continue
            
            data = loc_api_get(FSA_COLLECTION_URL, params={
                'dates': f'{year}/{year}',
                'c': RESULTS_PER_PAGE,
                'at': 'results,pagination',
                'sp': page
            })
            requests_since_save += 1
            
            if not data or 'results' not in data:
                print(f'  {year} str.{page}: brak danych, kończę ten rok')
                break
            
            results = data['results']
            if len(results) == 0:
                break
            
            # Spłaszcz + deduplikuj po id
            new_count = 0
            for rec in results:
                flat = flatten_record(rec)
                rec_id = flat.get('id', '')
                if rec_id and rec_id in progress['seen_ids']:
                    continue  # duplikat (nakładanie facetów)
                if rec_id:
                    progress['seen_ids'].add(rec_id)
                progress['all_keys'].update(flat.keys())
                buffer.append(flat)
                new_count += 1
            
            progress['done'].add((year, page))
            progress['total_records'] += new_count
            
            # Sprawdź czy ostatnia strona
            pag = data.get('pagination', {})
            total_pages = pag.get('total', page)
            
            print(f'  {year} str.{page}/{total_pages}: +{new_count} nowych '
                  f'(łącznie {progress["total_records"]})')
            
            # Zapis co SAVE_EVERY żądań
            if requests_since_save >= SAVE_EVERY:
                flush(progress, buffer)
                buffer = []
                requests_since_save = 0
                print(f'    💾 zapisano na Drive')
            
            if page >= total_pages:
                break
            page += 1
    
    # Zapisz resztę
    flush(progress, buffer)
    buffer = []
    print(f'\n🎉 KOMPLETNE! Pobrano {progress["total_records"]} unikalnych rekordów')
    print(f'   CSV: {CSV_PATH}')

except KeyboardInterrupt:
    print('\n⏸ Przerwano ręcznie. Zapisuję bufor...')
    flush(progress, buffer)
    print(f'   Zapisano. Uruchom komórkę ponownie by wznowić od {progress["total_records"]} rek.')
except Exception as e:
    print(f'\n✗ Błąd: {e}. Zapisuję bufor...')
    flush(progress, buffer)
    print(f'   Bufor zapisany. Można wznowić.')
    raise


## 4. Weryfikacja pobranego CSV


In [ ]:
if CSV_PATH.exists():
    df = pd.read_csv(CSV_PATH, dtype=str, keep_default_na=False)
    print(f'✅ CSV: {len(df)} rekordów, {len(df.columns)} kolumn\n')
    
    # Sprawdź że reproductions NIE ma
    if 'reproductions' in df.columns:
        print('⚠ UWAGA: kolumna reproductions wciąż obecna!')
    else:
        print('✓ reproductions pominięte (zgodnie z konfiguracją)')
    
    # Duplikaty
    if 'id' in df.columns:
        dups = df['id'].duplicated().sum()
        print(f'✓ Duplikaty po id: {dups}')
    
    print(f'\nKolumny ({len(df.columns)}):')
    for col in sorted(df.columns):
        non_empty = (df[col] != '').sum()
        print(f'  {col:28} : {non_empty:>6} ({non_empty*100//max(len(df),1)}%)')
else:
    print('CSV nie istnieje — uruchom scraper (komórka 3)')


## 5. (Opcjonalnie) Reset postępu

Jeśli chcesz zacząć od zera, odkomentuj i uruchom. **Usuwa CSV i postęp z Drive.**


In [ ]:
# UWAGA: to usuwa pobrane dane! Odkomentuj świadomie.

# if CSV_PATH.exists():
#     CSV_PATH.unlink()
#     print('Usunięto CSV')
# if PROGRESS_PATH.exists():
#     PROGRESS_PATH.unlink()
#     print('Usunięto postęp')
# print('Reset gotowy — następne uruchomienie scrapera zacznie od zera')
